## ToDo:

- implement delete function for shifts
- implement delete function for background (can it be the same?)
- write tests!
- add to tutorial

In [17]:
from ponds import PONDS, SIGMOID, ELLIPSIS, WhiteNoise
import matplotlib.pyplot as plt
import xarray as xr

# initialize PONDS object
pd = PONDS(
    lat_size=20,
    lon_size=40,
    time_size=200,
)

# add background with white noise
pd.add_background(
    background_method=WhiteNoise(
        magnitude=0.05,
        background_seed=2,
    )
)

pd_rep1 = xr.Dataset.copy(pd.data)

# add background with white noise
pd.add_background(
    background_method=WhiteNoise(
        magnitude=0.05,
    )
)

# add random shift
pd.add_shift(
    shape_method=ELLIPSIS(shape_seed=1),
    shift_method=SIGMOID(central_time=25),
)

pd_rep2 = xr.Dataset.copy(pd.data)

# add shift with custom parameters
shift_dict = {
    "shift_steepness": 0.2,
    "shift_magnitude": 0.5,
}
pd.add_shift(
    shape_method=ELLIPSIS(shape_seed=4),
    shift_method=SIGMOID(**shift_dict),
)

In [18]:
pd.data

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 200, lat: 20, lon: 40)
Coordinates:
  * time     (time) float64 2kB 0.0 1.0 2.0 3.0 4.0 ... 196.0 197.0 198.0 199.0
  * lat      (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon      (lon) float64 320B -180.0 -170.8 -161.5 ... 161.5 170.8 180.0
Data variables:
    ts       (time, lat, lon) float64 1MB 0.02673 0.1166 ... -0.1154 -0.04611
    c1       (time, lat, lon) bool 160kB False False False ... False False False
    c2       (time, lat, lon) bool 160kB False False False ... False False False
Attributes:
    cluster_key:     c
    background_key:  b

In [19]:
pd_rep1

<xarray.Dataset> Size: 1MB
Dimensions:  (time: 200, lat: 20, lon: 40)
Coordinates:
  * time     (time) float64 2kB 0.0 1.0 2.0 3.0 4.0 ... 196.0 197.0 198.0 199.0
  * lat      (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon      (lon) float64 320B -180.0 -170.8 -161.5 ... 161.5 170.8 180.0
Data variables:
    ts       (time, lat, lon) float64 1MB 0.02673 0.1166 ... -0.1154 -0.04611
Attributes:
    cluster_key:     c
    background_key:  b

In [20]:
# check if pd.data and pd_rep are identical (by content)
identical = pd.data.equals(pd_rep1)
print("identical (equals):", identical)

identical (equals): False


In [21]:
#pd.delete_shift(cluster="c1")
pd.delete_shift(cluster="c2")
pd.delete_background("b2")

In [22]:
# check if pd.data and pd_rep are identical (by content)
identical = pd.data.equals(pd_rep1)
print("identical (equals):", identical)

identical (equals): False


In [23]:
pd.data

<xarray.Dataset> Size: 1MB
Dimensions:  (time: 200, lat: 20, lon: 40)
Coordinates:
  * time     (time) float64 2kB 0.0 1.0 2.0 3.0 4.0 ... 196.0 197.0 198.0 199.0
  * lat      (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon      (lon) float64 320B -180.0 -170.8 -161.5 ... 161.5 170.8 180.0
Data variables:
    ts       (time, lat, lon) float64 1MB 0.009453 0.02516 ... -0.03786 -0.07343
    c1       (time, lat, lon) bool 160kB False False False ... False False False
Attributes:
    cluster_key:     c
    background_key:  b

In [24]:
c1_instance = pd.get_cluster_instance("c1")

display(c1_instance)

c1_instance[0].param_dict

(<ponds.shiftgen.SIGMOID at 0x760408ef71d0>,
 <ponds.correlationgen.NoCorr at 0x760408f017f0>)

{'shift_type': ponds.shiftgen.SIGMOID,
 'central_time': 25,
 'shift_steepness': 2.0425487000027958,
 'shift_magnitude': 1.0423534320342558}

In [18]:
print(np.all(pd.data.ts.values == 0)) 

True


In [37]:
import numpy as np
from ponds import Trend

# 1) setup variables
lat_size = 2
lon_size = 4
time_size = 2

# create first ponds object with two backgrounds
ponds1 = PONDS(
    lat_size=lat_size,
    lon_size=lon_size,
    time_size=time_size,
)

# add two backgrounds to first ponds object
ponds1.add_background(
    background_method=Trend(
    ),
    #shape_method=mock_shape(),
)
ponds1.add_background(
    background_method=WhiteNoise(
        background_seed=43,
    ),
    #shape_method=mock_shape(),
)

# delete the first background
ponds1.delete_background("b1")

# create second ponds object with only the second background
ponds2 = PONDS(
    lat_size=lat_size,
    lon_size=lon_size,
    time_size=time_size,
)
ponds2.add_background(
    background_method=WhiteNoise(
        background_seed=43,         # <- needs to be the same as in ponds1, b2
    ),
    #shape_method=mock_shape(),
)



# 2) check results
# - check if the time series of both ponds objects are identical
assert np.array_equal(ponds1.data.ts.values, ponds2.data.ts.values), (
    "Time series after deleting background do not match the expected time series."
)
# - check if the attributes related to the deleted background are removed
assert "b1_background_type" not in ponds1.data.ts.attrs, (
    "Attributes related to deleted background 'b1' were not removed."
)
assert "b1_shape_type" not in ponds1.data.ts.attrs, (
    "Attributes related to deleted background 'b1' were not removed."
)

AssertionError: Time series after deleting background do not match the expected time series.

In [38]:
ponds1.data

<xarray.Dataset> Size: 192B
Dimensions:  (time: 2, lat: 2, lon: 4)
Coordinates:
  * time     (time) float64 16B 0.0 1.0
  * lat      (lat) float64 16B -90.0 90.0
  * lon      (lon) float64 32B -180.0 -60.0 60.0 180.0
Data variables:
    ts       (time, lat, lon) float64 128B 0.01221 -0.02928 ... 0.001328
Attributes:
    cluster_key:     c
    background_key:  b

In [39]:
ponds2.data

<xarray.Dataset> Size: 192B
Dimensions:  (time: 2, lat: 2, lon: 4)
Coordinates:
  * time     (time) float64 16B 0.0 1.0
  * lat      (lat) float64 16B -90.0 90.0
  * lon      (lon) float64 32B -180.0 -60.0 60.0 180.0
Data variables:
    ts       (time, lat, lon) float64 128B 0.01221 -0.02928 ... 0.001328
Attributes:
    cluster_key:     c
    background_key:  b

In [45]:
# compare ponds1 and ponds2 time series and show where they differ
a = ponds1.data.ts.values
b = ponds2.data.ts.values

if a.shape != b.shape:
    print("Shape mismatch:", a.shape, b.shape)
else:
    diff_mask = ~np.equal(a, b)
    diff_indices = np.argwhere(diff_mask)
    n_diff = diff_indices.shape[0]
    print(f"Total differing elements: {n_diff} / {a.size}")

    if n_diff == 0:
        print("No differences found.")
    else:
        # get coordinate arrays for nicer reporting (if available)
        try:
            times = ponds1.data.coords["time"].values
            lats = ponds1.data.coords["lat"].values
            lons = ponds1.data.coords["lon"].values
        except Exception:
            times = lats = lons = None

        # show up to first 50 differences
        for i, (t, lat_i, lon_i) in enumerate(diff_indices[:50]):
            val1 = a[t, lat_i, lon_i]
            val2 = b[t, lat_i, lon_i]
            diff = val2 - val1
            if times is not None:
                coord_str = f"time={times[t]}, lat={lats[lat_i]}, lon={lons[lon_i]}"
            else:
                coord_str = f"index=({t},{lat_i},{lon_i})"
            print(f"{i+1}. {coord_str}: ponds1={val1:.6g}, ponds2={val2:.6g}, diff={diff:.6g}")

        if n_diff > 50:
            print(f"... and {n_diff-50} more differences.")

Total differing elements: 2 / 16
1. time=1.0, lat=-90.0, lon=180.0: ponds1=0.0102866, ponds2=0.0102866, diff=-1.73472e-18
2. time=1.0, lat=90.0, lon=-180.0: ponds1=0.0613249, ponds2=0.0613249, diff=6.93889e-18
